# fase_3 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 3.

**Purpose**: Migrasi Rekrutmen & Pelamar dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector 
import pandas as pd
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

Connected to dataleap_v5_example and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('pengajuan', 'pengajuan_karyawan'),
    ('histori_pengajuan', 'histori_pengajuan'),
    ('pelamar', 'pelamar'),
    ('pekerjaan', 'pelamar_kerja'),
    ('pendidikan', 'pelamar_sekolah'),
    ('kursus', 'pelamar_kursus'),
    ('pelamar_note', 'progres_pelamar'),
    ('pelamar_users', 'rekrutmen_pelamar')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    cursor_old.execute(f"SELECT * FROM `{old_t}`")
    raw_data[old_t] = cursor_old.fetchall()
    print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")

✅ pengajuan loaded: 33 records
✅ histori_pengajuan loaded: 79 records
✅ pelamar loaded: 178 records
✅ pekerjaan loaded: 67 records
✅ pendidikan loaded: 53 records
✅ kursus loaded: 50 records
✅ pelamar_note loaded: 403 records
✅ pelamar_users loaded: 281 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# Fungsi pembantu untuk pecah TTL
def extract_place(ttl):
    if pd.isna(ttl) or not str(ttl).strip(): return None
    return str(ttl).split(',')[0].strip() if ',' in str(ttl) else str(ttl)

def extract_date(ttl):
    if pd.isna(ttl) or not str(ttl).strip(): return None
    if ',' in str(ttl):
        try:
            return str(ttl).split(',')[1].strip()
        except:
            return None
    return None

# 1. pengajuan -> pengajuan_karyawan
if 'pengajuan' in raw_data:
    df = pd.DataFrame(raw_data['pengajuan'])
    mapping = {
        'idpengajuan': 'id_pengajuan', 'idusers': 'id_user', 'keterangan': 'posisi',
        'jumlah': 'jumlah', 'syarat': 'syarat', 'pertanyaan': 'pertanyaan',
        'alur': 'alur_seleksi', 'test': 'daftar_tes', 'status': 'status',
        'created_at': 'created_at'
    }
    transformed_dfs['pengajuan_karyawan'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 2. histori_pengajuan -> histori_pengajuan
if 'histori_pengajuan' in raw_data:
    df = pd.DataFrame(raw_data['histori_pengajuan'])
    mapping = {
        'idhistori': 'id_verifikasi', 'idpengajuan': 'id_pengajuan',
        'status': 'status_verifikasi_pengajuan', 'catatan': 'catatan',
        'created_at': 'created_at'
    }
    transformed_dfs['histori_pengajuan'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 3. pelamar -> pelamar
if 'pelamar' in raw_data:
    df = pd.DataFrame(raw_data['pelamar'])
    # Pecah TTL
    df['tempat_lahir'] = df['ttl'].apply(extract_place)
    df['tanggal_lahir'] = df['ttl'].apply(extract_date)
    
    mapping = {
        'idpelamar': 'id_pelamar', 'idpengajuan': 'id_pengajuan', 'email': 'email_pelamar',
        'nama': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jk': 'jenis_kelamin',
        'tempat_lahir': 'tempat_lahir', 'tanggal_lahir': 'tanggal_lahir',
        'alamat': 'alamat_ktp', 'domisili': 'alamat_domisili', 'wa': 'nomor_wa',
        'linkedin': 'akun_linkedin', 'ig': 'akun_instagram', 'fb': 'akun_facebook', 
        'sosmed': 'sosmed_lain', 'laptop': 'spesifikasi_laptop', 'internet': 'internet',
        'kegiatan': 'kegiatan_sekarang', 'rencana': 'rencana_karir', 'mobilitas': 'mobilitas',
        'info': 'sumber_info', 'wfo': 'siap_wfo', 'bergabung': 'tanggal_bergabung',
        'jenis': 'kategori_pelamar', 'work': 'riwayat_kerja', 'ppdk': 'riwayat_pendidikan',
        'pengalaman': 'pengalaman_bidang', 'wawasan': 'wawasan', 'sehat': 'riwayat_kesehatan',
        'statusnikah': 'status_pernikahan', 'ajar': 'kemampuan_ajar', 'app': 'penguasaan_aplikasi', 
        'apps': 'aplikasi_lainnya', 'gunalaptop': 'penggunaan_laptop', 'toefl': 'skor_toefl',
        'gaji': 'ekspektasi_gaji', 'link': 'tautan_berkas', 'resign': 'alasan_resign',
        'hasiliq': 'skor_iq', 'piciq': 'foto_iq', 'picminat': 'foto_minat', 
        'picpribadi': 'foto_kepribadian', 'created_at': 'created_at'
    }
    transformed_dfs['pelamar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 4. pekerjaan -> pelamar_kerja
if 'pekerjaan' in raw_data:
    df = pd.DataFrame(raw_data['pekerjaan'])
    mapping = {
        'idpekerjaan': 'id_pelamar_kerja', 'idusers': 'id_pelamar',
        'namaperusahaan': 'nama_perusahaan', 'periode': 'periode', 'jabatan': 'jabatan',
        'jobdesk': 'deskripsi_kerja'
    }
    transformed_dfs['pelamar_kerja'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 5. pendidikan -> pelamar_sekolah
if 'pendidikan' in raw_data:
    df = pd.DataFrame(raw_data['pendidikan'])
    mapping = {
        'idpendidikan': 'id_pelamar_sekolah', 'idusers': 'id_pelamar',
        'sekolah': 'nama_sekolah', 'jenjang': 'jenjang', 'prodi': 'prodi',
        'tahun': 'tahun_lulus', 'ipk': 'ipk', 'organisasi': 'organisasi'
    }
    transformed_dfs['pelamar_sekolah'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 6. kursus -> pelamar_kursus
if 'kursus' in raw_data:
    df = pd.DataFrame(raw_data['kursus'])
    mapping = {
        'idkursus': 'id_pelamar_kursus', 'idusers': 'id_pelamar',
        'nama': 'nama_kursus', 'tanggal': 'tanggal', 'deskripsi': 'deskripsi',
        'lokasi': 'lokasi', 'nosertifikat': 'nomor_sertifikat'
    }
    transformed_dfs['pelamar_kursus'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 7. pelamar_note -> progres_pelamar
if 'pelamar_note' in raw_data:
    df = pd.DataFrame(raw_data['pelamar_note'])
    mapping = {
        'idnote': 'id_progres_pelamar', 'idpelamar': 'id_pelamar',
        'idusers': 'id_user', 'status': 'status_progres_pelamar',
        'note': 'catatan', 'link': 'tautan_file', 'pertanyaan': 'pertanyaan',
        'created_at': 'created_at'
    }
    transformed_dfs['progres_pelamar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 8. pelamar_users -> rekrutmen_pelamar
if 'pelamar_users' in raw_data:
    df = pd.DataFrame(raw_data['pelamar_users'])
    mapping = {
        'idassign': 'id_rekrutmen', 'idpelamar': 'id_pelamar', 'idusers': 'id_user'
    }
    transformed_dfs['rekrutmen_pelamar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 3 selesai.")

✓ Transformasi 8 tabel Fase 3 selesai.


## 4. Export ke Pickle

In [5]:
file_name = 'fase_3_hanif.pkl'
with open(file_name, 'wb') as f:
    pickle.dump(transformed_dfs, f)

total_records = sum(len(df) for df in transformed_dfs.values())
migration_result = {
    'fase': 'fase_3',
    'script': 'script_hanif',
    'fase_num': 3,
    'status': 'ready_for_insert',
    'records_transformed': total_records,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_3",
  "script": "script_hanif",
  "fase_num": 3,
  "status": "ready_for_insert",
  "records_transformed": 1144,
  "pickle_file": "fase_3_hanif.pkl",
  "timestamp": "2026-04-28T16:48:55.577152"
}
